In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260530_142928"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))
trades = pd.read_parquet(os.path.join(folder_path, "trades.parquet"))
quotes = pd.read_parquet(os.path.join(folder_path, "quotes.parquet"))
fills = pd.read_parquet(os.path.join(folder_path, "fills.parquet"))

events = pd.read_parquet(os.path.join(folder_path, "events.parquet"))
with open(os.path.join(folder_path, "orderbook_snapshot.json"), "r") as f:
    orderbook_snapshot = json.load(f)

In [3]:
df = snapshots
df = df.sort_values("ts").reset_index(drop=True)

df["ts"] = pd.to_datetime(df["ts"])
df = df.set_index("ts")

In [ ]:
"""
2. Choose regime window (critical design choice)
Start simple:
"""

WINDOW = "1s"   # later try 2s, 5s

In [ ]:
"""
3. Build core regime features (windowed)

This is your regime feature matrix.
"""

regime_df = pd.DataFrame()

In [ ]:
"""3.1 Volatility regime"""
regime_df["volatility"] = df["mid"].pct_change().rolling(WINDOW).std()

"""3.2 Spread regime"""
regime_df["spread"] = df["spread"].rolling(WINDOW).mean()

"""3.3 Order flow regime"""
regime_df["order_imbalance"] = df["order_imbalance"].rolling(WINDOW).mean()
regime_df["trade_imbalance"] = df["trade_imbalance"].rolling(WINDOW).mean()

regime_df["quote_churn"] = df["quote_churn"].rolling(WINDOW).mean()

"""3.5 Inventory stress (important for MM regimes)"""
regime_df["inventory"] = df["inventory"].rolling(WINDOW).mean()
regime_df["inventory_vol"] = df["inventory"].rolling(WINDOW).std()

"""3.6 Price efficiency / microprice divergence

This is VERY important for regimes:
"""

regime_df["microprice_error"] = (
    df["mid"] - df["microprice"]
).rolling(WINDOW).mean()


"""4. Add execution quality (if aligned fills exist)

If you merged fills into rows:
"""
regime_df["avg_markout"] = df.get("markout_1000ms", np.nan).rolling(WINDOW).mean()
regime_df["toxicity"] = df.get("toxic", 0).rolling(WINDOW).mean()


"""

1. Regime model (clustering / HMM / classifier)

This is your:

regime_model.fit(X_regime)

👉 Labeling happens AFTER this step

Flow:

Step 1 — build features (NO labels yet)

You compute:

volatility
spread
imbalance
etc.
Step 2 — train regime model (unsupervised)
X_regime → clustering model → regime_id
Step 3 — THEN you label regimes

You do:

df.groupby("regime").agg({... future_return ...})

👉 This is post-training interpretation"""

In [ ]:
"""
5. Clean dataset
regime_df = regime_df.dropna()

6. Standardize features (VERY important for clustering)
from sklearn.preprocessing import StandardScaler
"""

scaler = StandardScaler()
X = scaler.fit_transform(regime_df.values)

regime_X = pd.DataFrame(
    X,
    index=regime_df.index,
    columns=regime_df.columns
)

In [ ]:
n_regimes = 3  # start small: 2–5 max

model = GaussianMixture(n_components=n_regimes, covariance_type="full", random_state=42)

regime_labels = model.fit_predict(regime_X)

regime_df["regime"] = regime_labels

In [ ]:
def detect_regime(self, features):
    x = np.array([
        features["volatility"],
        features["spread"],
        features["order_imbalance"],
        features["trade_imbalance"],
        features["quote_churn"],
        features["inventory"],
    ]).reshape(1, -1)

    x = self.scaler.transform(x)

    regime = self.regime_model.predict(x)[0]

    return regime

In [ ]:
"""
9. What you get from this immediately

You can now compute:

For each regime:
"""

regime_stats = regime_df.groupby("regime").agg({
    "volatility": "mean",
    "spread": "mean",
    "microprice_error": "mean",
    "toxicity": "mean",
})

"""
This tells you:

which regime is toxic
which is profitable for MM
where alpha works best
"""

In [ ]:
import pandas as pd
import numpy as np

def add_evaluation_labels(self, horizon=10):
    """
    Adds forward-looking evaluation labels to each snapshot row.
    These are used to interpret regimes AFTER clustering.
    """

    df = pd.DataFrame(self.rows)
    df = df.sort_values("ts").reset_index(drop=True)

    mid = df["mid"].values

    n = len(df)

    future_return = np.full(n, np.nan)
    future_volatility = np.full(n, np.nan)
    future_direction = np.full(n, np.nan)

    for i in range(n - horizon):

        p0 = mid[i]
        p1 = mid[i + horizon]

        window = mid[i:i + horizon]

        # 1. Return (trend / drift)
        future_return[i] = (p1 - p0) / p0

        # 2. Realized volatility in future window
        future_volatility[i] = np.std(np.diff(window) / window[:-1])

        # 3. Direction (simple sign regime)
        future_direction[i] = np.sign(p1 - p0)

    df["future_return"] = future_return
    df["future_volatility"] = future_volatility
    df["future_direction"] = future_direction

    self.regime_eval_df = df
"""
3. Add microstructure “regime quality” labels

These are VERY important for MM regimes.

(A) Microprice error (efficiency)
"""
df["future_microprice_error"] = (
    df["mid"].shift(-horizon) - df["microprice"]
) / df["microprice"]

"""
(B) Spread regime stability
"""

df["future_spread_mean"] = (
    df["spread"].rolling(horizon).mean().shift(-horizon)
)

"""
(C) Inventory stress proxy (optional but powerful)
"""

df["future_inventory_vol"] = (
    df["inventory"].rolling(horizon).std().shift(-horizon)
)

"""
(D) Toxicity proxy (if you don’t have fill alignment yet)
"""

Approximate:

df["future_abs_return"] = np.abs(df["future_return"])
df["trend_strength"] = df["future_return"] / (df["future_volatility"] + 1e-9)

"""
4. Final evaluation dataset
"""
self.regime_eval_df = df.dropna()

"""
After clustering:
"""

df.groupby("regime").agg({
    "future_return": "mean",
    "future_volatility": "mean",
    "trend_strength": "mean",
    "future_microprice_error": "mean"
})

"""
Now you can label:

Example interpretation:
high future return → trending regime
high future volatility → stress regime
high microprice error → inefficient regime (alpha-rich)
low everything → stable MM regime
"""

In [ ]:
# NEW

# RAW DATA
#    ↓
# REGIME FEATURES (no future)
#    ↓
# FIT SCALER + GMM
#    ↓
# ASSIGN REGIMES
#    ↓
# BUILD FUTURE LABELS (separate)
#    ↓
# JOIN ON TIME INDEX
#    ↓
# REGIME STATISTICS / LABELING

In [ ]:
# STEP 1 — Load raw data

df = snapshots
df = df.sort_values("ts").reset_index(drop=True)

df["ts"] = pd.to_datetime(df["ts"])
df = df.set_index("ts")

In [ ]:
# STEP 2 — Build REGIME FEATURES (ONLY past info) slower trends - 1000ms

regime_df = pd.DataFrame()

regime_df["volatility"] = df["mid"].pct_change().rolling("1s").std()
regime_df["spread"] = df["spread"].rolling("1s").mean()
regime_df["order_imbalance"] = df["order_imbalance"].rolling("1s").mean()
regime_df["trade_imbalance"] = df["trade_imbalance"].rolling("1s").mean()
regime_df["quote_churn"] = df["quote_churn"].rolling("1s").mean()
regime_df["inventory"] = df["inventory"].rolling("1s").mean()
regime_df["inventory_vol"] = df["inventory"].rolling("1s").std()
regime_df["microprice_error"] = (df["mid"] - df["microprice"]).rolling("1s").mean()

regime_df = regime_df.dropna()

In [ ]:
# STEP 3 — Train regime model

scaler = StandardScaler()
X = scaler.fit_transform(regime_df)

model = GaussianMixture(n_components=3)
regime_df["regime"] = model.fit_predict(X)

In [ ]:
# STEP 4 — NOW build evaluation labels (separate dataset)

eval_df = df.copy()
mid = eval_df["mid"].values
h = 10 # 1000ms window

future_return = np.full(len(df), np.nan)  # creates an array the same length as your dataset and fills every element with NaN 
future_volatility = np.full(len(df), np.nan)
future_direction = np.full(len(df), np.nan)

for i in range(len(df) - h):

    p0 = mid[i]
    p1 = mid[i + h]

    window = mid[i:i + h]

    # 1. Return (trend / drift)
    future_return[i] = (p1 - p0) / p0

    # 2. Realized volatility in future window
    future_volatility[i] = np.std(np.diff(window) / window[:-1])

    # 3. Direction (simple sign regime)
    future_direction[i] = np.sign(p1 - p0)

# 👇 INSERT HERE (this is the key step)
eval_df["future_return"] = future_return
eval_df["future_volatility"] = future_volatility
eval_df["future_direction"] = future_direction 

# Future direction regime
# If result ≈ +1
# almost always up moves after this regime
# strong bullish bias
# If result ≈ -1
# almost always down moves after this regime
# bearish bias
# If result ≈ 0
# no directional bias
# pure noise / mean reversion / stable

# optional cleanup AFTER labeling
eval_df = eval_df.dropna(subset=["future_return", "future_volatility", "future_direction"])

In [ ]:
# STEP 5 — ALIGN BOTH DATASETS

# This is the missing step in your code.

# Now regime + outcome are aligned.

final = regime_df.join(
    eval_df[[
        "future_return",
        "future_volatility",
        "future_direction",
        "mid",
        "spread",
        "microprice"
    ]],
    how="inner"
)

In [ ]:
# STEP 6 — ANALYZE REGIMES

final.groupby("regime").agg({
    "future_return": "mean",
    "spread": "mean",
    "microprice_error": "mean",
    "future_direction": "mean"
})

# Research phase

# There is some human interpretation.

# You might see:

# regime	future_return	future_volatility
# 0	0.0000	0.0004
# 1	0.0001	0.0030
# 2	0.0015	0.0005

# and conclude:

# 0 = STABLE_MM
# 1 = HIGH_VOL
# 2 = TRENDING

# That's "eyeballing", but it's not arbitrary. You're looking at measurable statistics.

In [ ]:
# 5. Your detect_regime() function is fine BUT incomplete

# You MUST ensure:

# same feature order
# same scaler trained on regime_df
# no missing values

def detect_regime(self, features):
    x = np.array([
        features["volatility"],
        features["spread"],
        features["order_imbalance"],
        features["trade_imbalance"],
        features["quote_churn"],
        features["inventory"],
    ]).reshape(1, -1)

    x = self.scaler.transform(x)
    return self.regime_model.predict(x)[0]